In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')
!pip install mne
import os
import mne
import matplotlib.pyplot as plt
import gc

In [ ]:
# Root directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Dict of components to exclude per participant
ica_removal_dict = {
    "3": [13],
    "6": [12, 14],
    "7": [8, 18],
    "8": [8],
    "10": [15],
    "11": [],
    "12": [17],
    "13": [9, 15, 16, 11, 17,18],
    "14": [7],
    "15": [0, 2, 11],
    "17": [2, 3, 6, 12, 16, 19],
    "18": [0, 8, 12, 13, 17],
    "19": [0, 2, 7, 8, 9, 15, 16],
    "20": [],
    "21": [3, 15],
    "22": [0, 1, 3, 6, 13],
    "24": [5],
    "25": [12, 17, 18],
    "26": [4],
    "27": [15]
}

# Loop through participants
for pid, comps in ica_removal_dict.items():
    print(f"\n>>> Processing participant {pid} | Exclude components: {comps}")
    participant_path = os.path.join(root_dir, pid)

    raw_path = os.path.join(participant_path, f"{pid}_filtered_raw.fif")
    ica_path = os.path.join(participant_path, f"{pid}_ica.fif")
    cleaned_path = os.path.join(participant_path, f"{pid}_clean_after_ica.fif")

    # Skip if output already exists
    if os.path.exists(cleaned_path):
        print(f"[{pid}] ✅ Already cleaned — skipping.")
        continue

    if not os.path.exists(raw_path):
        print(f"[{pid}] ❌ Raw file not found: {raw_path}")
        continue
    if not os.path.exists(ica_path):
        print(f"[{pid}] ❌ ICA file not found: {ica_path}")
        continue

    try:
        # Load raw and ICA
        raw = mne.io.read_raw_fif(raw_path, preload=True)
        ica = mne.preprocessing.read_ica(ica_path)

        # Apply ICA exclusions
        ica.exclude = comps
        raw_clean = ica.apply(raw.copy())
        print(f"[{pid}] ✓ ICA applied")

        # Plot time series of sources (after ICA)
        fig_sources = ica.plot_sources(raw_clean, show=False)
        fig_path = os.path.join(participant_path, f"{pid}_ica_sources_removed.png")
        fig_sources.savefig(fig_path, dpi=150)
        plt.close(fig_sources)
        print(f"[{pid}] ✓ Time series plot saved: {fig_path}")

        # Save cleaned raw
        raw_clean.save(cleaned_path, overwrite=True)
        print(f"[{pid}] ✓ Cleaned raw saved: {cleaned_path}")

        # Cleanup
        del raw, raw_clean, ica, fig_sources
        gc.collect()

    except Exception as e:
        print(f"[{pid}] ⚠️ Error: {e}")
